# 🔎 Research Assistant Agent: Web Search, Page Reading and Weather

## Learning Objectives
In this notebook, you will learn:
1. **Tools for a research assistant** - Wrap Tavily search, full-page extraction (MarkItDown) and the OpenWeatherMap API as tools, with limits that keep tool output affordable
2. **Checking tool choice before building the agent** - Confirm the model picks the right tool for each kind of question
3. **Assembling the assistant with `create_agent`** - Compare a bare LLM with the tool-equipped agent on a question that needs fresh data
4. **Conversation memory** - Show the stateless agent failing a follow-up question, then fix it with a checkpointer and a `thread_id`, checked against the tool data

## Where this fits
Modules 02–04 taught the mechanics this notebook relies on: defining tools (`02_tools_and_function_calling/01_defining_tools.ipynb`), the ReAct loop (`03_react_agents/01_react_agent_langgraph.ipynb`) and `create_agent` (`03_react_agents/03_react_with_create_agent.ipynb`). Here they are put together into one small, complete application: a research assistant that searches the web, reads the pages it finds, checks the weather and remembers the conversation.

## Prerequisites
- Tools, the ReAct loop and `create_agent` from Modules 02–04 (reading only)
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY`, `TAVILY_API_KEY` and `WEATHER_API_KEY` (an [OpenWeatherMap](https://openweathermap.org/api) key) in the project-root `.env`

---
## 📦 Step 1: Environment Setup

Load the API keys from the project-root `.env`.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY, TAVILY_API_KEY and WEATHER_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: the reasoning engine of the research assistant
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

---
## 🧰 Part 1: The Research Tools

A research assistant needs two kinds of information that a model does not have: **fresh web content** and **live data** such as the current weather. Each becomes a tool.

### Key Concepts:
- **Search + extract**: Tavily returns the most relevant URLs with snippets; MarkItDown then fetches each page and converts it to plain text, so the model reads the page rather than a two-line snippet
- **Budget the tool output**: every character a tool returns is sent back to the model. Full web pages can be tens of thousands of tokens each, so the tool keeps `MAX_PAGES` pages and cuts each to `MAX_CHARS_PER_PAGE` characters
- **Tool errors as messages**: when a page times out or a city is unknown, the tool returns a short explanation instead of raising, so the agent can adapt

> **Key Insight**: The tool's docstring is its user manual for the model — it decides *when* the tool is used. The limits inside the tool decide what each call *costs*.

In [ ]:
# ============================================================================
# TOOL 1: Web search + page extraction
# ============================================================================
import os
from concurrent.futures import ThreadPoolExecutor
from concurrent.futures import TimeoutError as FutureTimeout

import requests
from langchain_core.tools import tool
from langchain_tavily import TavilySearch
from markitdown import MarkItDown

MAX_PAGES = 3               # pages read per search
MAX_CHARS_PER_PAGE = 4000   # characters kept from each page
PAGE_TIMEOUT_S = 15         # give up on slow sites

tavily_search = TavilySearch(max_results=MAX_PAGES, search_depth="advanced")

# A browser-like session: some sites refuse the default Python user agent
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
})
md_converter = MarkItDown(requests_session=session)


def _read_page(url: str) -> str:
    """Fetch one URL and return its title + text, cut to MAX_CHARS_PER_PAGE."""
    page = md_converter.convert(url)
    text = f"{(page.title or '').strip()}\n{page.text_content.strip()}"
    return text[:MAX_CHARS_PER_PAGE]


@tool
def search_web_extract_info(query: str) -> list[str]:
    """Search the web for a query and return the text of the most relevant pages.
    Use it for recent events, facts you are unsure about, or anything after your training data."""
    hits = tavily_search.invoke({"query": query})["results"]
    docs = []
    with ThreadPoolExecutor() as pool:
        futures = {hit["url"]: pool.submit(_read_page, hit["url"]) for hit in hits}
        for hit in hits:
            url = hit["url"]
            try:
                docs.append(f"SOURCE: {url}\n{futures[url].result(timeout=PAGE_TIMEOUT_S)}")
            except FutureTimeout:
                docs.append(f"SOURCE: {url}\n(timed out; snippet) {hit['content']}")
            except Exception as exc:  # blocked page, PDF parse error, ...
                docs.append(f"SOURCE: {url}\n(could not read page: {type(exc).__name__}; snippet) {hit['content']}")
    return docs


print(f"✅ search_web_extract_info ready ({MAX_PAGES} pages × ≤{MAX_CHARS_PER_PAGE} chars)")

In [ ]:
# ============================================================================
# TOOL 2: Current weather from OpenWeatherMap
# ============================================================================
@tool
def get_weather(city: str) -> dict | str:
    """Get the current weather for a city (temperature in °C, humidity, wind, conditions).
    Use it whenever the user asks about the weather in a place."""
    resp = requests.get(
        "https://api.openweathermap.org/data/2.5/weather",
        params={"q": city, "appid": os.getenv("WEATHER_API_KEY"), "units": "metric"},
        timeout=15,
    )
    data = resp.json()
    if resp.status_code != 200:
        return f"Weather data not found for '{city}': {data.get('message', resp.status_code)}"
    # Keep only the fields a person would ask about (the raw payload is ~40 fields)
    return {
        "city": data["name"],
        "country": data["sys"]["country"],
        "conditions": data["weather"][0]["description"],
        "temp_c": data["main"]["temp"],
        "feels_like_c": data["main"]["feels_like"],
        "humidity_pct": data["main"]["humidity"],
        "wind_m_s": data["wind"]["speed"],
    }


# Try both tools directly — no LLM involved yet
print(get_weather.invoke({"city": "Bangalore"}))
print(get_weather.invoke({"city": "Qwertyville"}))
pages = search_web_extract_info.invoke({"query": "LangGraph 1.0 release"})
print(f"\n{len(pages)} pages, {sum(len(p) for p in pages):,} characters in total")
print(pages[0][:300], "...")

Notice the two weather results: a real city returns a compact dictionary, an unknown one returns a sentence the model can read. The search tool returns at most `MAX_PAGES × MAX_CHARS_PER_PAGE` characters, whatever the pages' real size.

---
## 🎯 Part 2: Checking Tool Choice

Before wrapping the tools in an agent, check that the model routes each kind of question to the right tool. `bind_tools` gives the model the tool schemas; its reply's `tool_calls` show what it *would* call. No tool runs here, so this check is cheap.

In [ ]:
# ============================================================================
# TOOL-CHOICE CHECK: one question per expected tool (+ one that needs none)
# ============================================================================
tools = [search_web_extract_info, get_weather]
llm_with_tools = llm.bind_tools(tools)

checks = [
    ("Summarize Microsoft's most recent quarterly earnings call", "search_web_extract_info"),
    ("How is the weather in Bangalore today?", "get_weather"),
    ("What is 12 multiplied by 12?", None),  # the model should answer directly
]

for question, expected in checks:
    calls = llm_with_tools.invoke(question).tool_calls
    chosen = calls[0]["name"] if calls else None
    status = "✅" if chosen == expected else "❌"
    print(f"{status} {question!r}\n    → {chosen}  {calls[0]['args'] if calls else ''}")

---
## 🤖 Part 3: Assembling the Research Assistant

`create_agent` wraps the model and the tools in the ReAct loop from Module 03: call the model, run any tools it asks for, feed the results back, stop when it answers without a tool call. The system prompt tells it how to behave as a research assistant.

First, the same question to the bare model and to the agent: the bare model can only answer from its training data.

In [ ]:
# ============================================================================
# CREATE THE AGENT
# ============================================================================
from datetime import date

from langchain.agents import create_agent

# The model does not know today's date; without it, "most recent" means "most recent in my training data"
SYS_PROMPT = f"""You are a helpful research assistant. Today's date is {date.today():%B %d, %Y}.
- For facts that may have changed recently (news, earnings, releases), search the web first.
- For weather questions, use the weather tool.
- Answer from the tool results, cite the source URLs you used, and say so when the tools
  did not return what was asked for."""

research_agent = create_agent(llm, tools, system_prompt=SYS_PROMPT)

print(f"✅ Agent graph nodes: {list(research_agent.get_graph().nodes)}")

In [ ]:
# ============================================================================
# BARE LLM vs AGENT on a question that needs fresh data
# ============================================================================
from IPython.display import Markdown, display

QUESTION = "Summarize the key points of Nvidia's most recent quarterly earnings call."

print("=== Bare LLM (training data only) ===")
print(llm.invoke(QUESTION).content[:800])


def tools_used(result) -> list[str]:
    """Names of the tools the agent called, in order."""
    return [tc["name"] for m in result["messages"] for tc in getattr(m, "tool_calls", [])]


result = research_agent.invoke({"messages": [{"role": "user", "content": QUESTION}]})
print(f"\n=== Agent (tools used: {tools_used(result)}) ===")
display(Markdown(result["messages"][-1].content))

The bare model can only describe what earnings calls usually cover (or an old quarter); the agent searches, reads the pages and cites them. The date in the system prompt matters: without it the model searches for "the most recent" quarter *of its training data*.

---
## 🧠 Part 4: Conversation Memory

Each `invoke` above started from an empty message list, so the agent has **no memory** between questions. A research conversation is full of follow-ups ("and Dubai?", "which is hotter?") that only make sense with the earlier turns.

### Key Concepts:
- **Checkpointer**: `InMemorySaver` stores the agent's state after every step, keyed by a `thread_id`
- **Thread**: every `invoke` with the same `{"configurable": {"thread_id": ...}}` continues the same conversation; a new id starts a fresh one

We ask about two cities, then the follow-up — first without memory, then with it — and check the answer against the temperatures the weather tool returned.

In [ ]:
# ============================================================================
# WITHOUT MEMORY: the follow-up has nothing to refer to
# ============================================================================
FOLLOW_UP = "Which of the two cities is hotter right now? Start your answer with the city name."

stateless = research_agent.invoke({"messages": [{"role": "user", "content": FOLLOW_UP}]})
print("Tools used:", tools_used(stateless))
print(stateless["messages"][-1].content)

Without the earlier turns the agent cannot know which two cities you mean. It either asks — or, as often happens, picks two cities of its own and answers confidently. Either way it is not answering *this* conversation.

In [ ]:
# ============================================================================
# WITH MEMORY: a checkpointer + thread_id carries the conversation
# ============================================================================
from langgraph.checkpoint.memory import InMemorySaver

memory_agent = create_agent(llm, tools, system_prompt=SYS_PROMPT, checkpointer=InMemorySaver())
thread = {"configurable": {"thread_id": "weather-chat-1"}}

turns = ["How is the weather in Bangalore right now?", "And in Dubai?", FOLLOW_UP]
for turn in turns:
    out = memory_agent.invoke({"messages": [{"role": "user", "content": turn}]}, thread)
    print(f"👤 {turn}\n🤖 {out['messages'][-1].content}\n")

print(f"Messages stored in the thread: {len(out['messages'])}")

In [ ]:
# ============================================================================
# CHECK THE ANSWER AGAINST THE TOOL DATA
# ============================================================================
import json

from langchain_core.messages import ToolMessage

# The city names the agent asked for, keyed by tool-call id
asked = {tc["id"]: tc["args"]["city"] for m in out["messages"]
         for tc in getattr(m, "tool_calls", []) if tc["name"] == "get_weather"}

# Temperatures exactly as the weather tool returned them in this thread
# (keyed by the name the user used, e.g. "Bangalore", not the API's "Bengaluru")
temps = {}
for m in out["messages"]:
    if isinstance(m, ToolMessage) and m.name == "get_weather":
        try:
            temps[asked[m.tool_call_id]] = json.loads(m.content)["temp_c"]
        except (json.JSONDecodeError, KeyError):
            pass

print("Tool data:", temps)
hotter = max(temps, key=temps.get)
answer = out["messages"][-1].content
first_line = answer.strip().splitlines()[0]
print(f"Hotter according to the tool data: {hotter}")
print(f"Answer's first line: {first_line!r}")
print("✅ answer matches the tool data" if hotter.lower() in first_line.lower()
      else "❌ answer disagrees with the tool data")

The last answer is correct only because the agent can see the two earlier tool results in the thread. The check reads those results from the stored messages instead of trusting the model's arithmetic.

---
## 📝 Summary

In this notebook, we learned:

### 1. Tools for a research assistant
- Search + page extraction gives the model full pages, not snippets; per-page and per-call limits keep the cost of each tool call bounded
- Tools return readable error messages (unknown city, unreadable page) instead of crashing the run

### 2. Checking tool choice
- `bind_tools` + `tool_calls` shows which tool the model would pick without running anything — a cheap check before building the agent

### 3. The assistant
- `create_agent` + a short system prompt turns the model and tools into a research assistant that answers questions the bare model cannot

### 4. Conversation memory
- Without a checkpointer every call is a new conversation; `InMemorySaver` + `thread_id` makes follow-up questions work
- Verify answers against the tool data stored in the thread, not against the model's own summary

### Next Steps
- **`02_competitive_intelligence_agent.ipynb`**: a domain-specific research agent that returns a structured market brief and checks that every cited source came from its own searches